In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ sigmoid(z) = p = \frac{e^z}{e^z+1} = \frac{1}{1+e^{-z}} $$
$$ \frac{\partial p}{\partial z} = \frac{e^z}{(e^z+1)^2} = p(1-p) $$


In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from common import assert_eq, T # type: ignore

def _sigmoid_neg(x: tr.Tensor) -> tr.Tensor:
    exp = tr.exp(x)
    return exp / (exp + 1)

def _sigmoid_pos(x: tr.Tensor) -> tr.Tensor:
    return 1 / (1 + tr.exp(-x))

def sigmoid(x) -> tr.Tensor:
    x = T(x)

    z = tr.empty_like(x)
    pos = x >= 0
    
    z[pos] = _sigmoid_pos(x[pos])
    z[~pos] = _sigmoid_neg(x[~pos])
    
    return z


class SigmoidFunction(autograd.Function):
    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        s = sigmoid(x)  
        ctx.save_for_backward(s)
        return s

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, ]:
        (s, ) = ctx.saved_tensors

        df_dx = s * (1 - s)
        assert_eq(df_dx.shape, s.shape)

        dF_dx = dF_df * df_dx
        assert_eq(dF_dx.shape, s.shape)

        return (dF_dx, )
    

class Sigmoid(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(x)


def test_sigmoid_1() -> None:
    assert_eq(sigmoid(1000), 1.0)
    assert_eq(sigmoid(-1000), 0.0)
    assert_eq(sigmoid(0), 0.5)
    assert_eq(sigmoid(1), 0.731, 0.001)
    assert_eq(sigmoid(-1), 0.269, 0.001)


def test_sigmoid_2() -> None:
    tr.manual_seed(0)

    samples = 10
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = SigmoidFunction.apply(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.sigmoid(x2)
    F2 = y2.sum()
    F2.backward()

    assert_eq(y1, y2, atol=0.001)
    assert_eq(x1.grad, x2.grad, atol=0.001)


if __name__ == "__main__":
    test_sigmoid_1()
    test_sigmoid_2()
